In [1]:
!pip install bertopic
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 81.0 MB/s eta 0:00:00


In [3]:
import pandas as pd
from bertopic import BERTopic
from bertopic.backend import MultiModalBackend
from bertopic.representation import KeyBERTInspired
import re
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
from hdbscan import HDBSCAN
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import random
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary
from textblob import TextBlob

In [8]:
df = pd.read_csv("/content/vibe_coding_combined_translated.csv")

In [9]:
df = df[["full_text_translated", "image_url"]]

In [10]:
def preprocess_tweet(text):
    #lower case text
    text = text.lower()
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Remove user mentions
    text = re.sub(r'@\w+', '', text)
    # Remove hashtags
    text = re.sub(r'#\w+', '', text)
    #remove symbols
    text = re.sub(r'[^\w\s]', '', text)
    #remove numbers
    text = re.sub(r'\d+', '', text)
    #Replace all "vibe code" into vibecode
    text = re.sub(r'vibe code', 'vibecode', text)
    #Replace all "vibe coding" into vibecoding
    text = re.sub(r'vibe coding', 'vibecoding', text)
    #Relace all "vibe coded" into vibecoded
    text = re.sub(r'vibe coded', 'vibecoded', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [11]:
df_preprocessed = df['full_text_translated'].fillna('').apply(preprocess_tweet)
df_preprocessed = pd.concat([df_preprocessed, df['image_url']], axis=1)

In [26]:
df_preprocessed = df_preprocessed[df_preprocessed['full_text_translated'].apply(lambda x: len(x.split()) >= 4)]
df_preprocessed.count()

,0
full_text_translated,20511
image_url,6495


In [59]:
docs = df_preprocessed['full_text_translated'].tolist()
images  = df_preprocessed['image_url'].tolist()
for i in range(len(images)):
    if pd.isna(images[i]):
        images[i] = None
random.shuffle(docs)
docs[:10]

['having thorough logging makes vibecoding x more productive',
 'i was antivibecoding but after using claude code sonnet the world has really started to change',
 'vibecoding is so last year for my followers follow raddka feel the future',
 'vibecoding is handsdown the best way to monetize your talents before i had to spend ages hunting for decent developers just to bring project ideas to life for various ecosystems now even my friend who draws nfts and had never touched code is launching highquality',
 'the number one tip i can give revibecoding vibeprdtestbuildvibe your llm doesnt understand your goalsit just makes pretty code that breaks when you blink',
 'my reaction when my prompt bangs out a great one shot whilst vibecoding',
 'if you read this original blog it will be helpful because i am having a hard time with things like deployment but eventually saas will become vibecoding and monetization will be integrated all at once',
 'the new gemini pro looks truly amazing for vibecodi

In [62]:
# embedding_model = SentenceTransformer("all-mpnet-base-v2")

umap_model = UMAP(n_neighbors=10, n_components=5, min_dist=0.0)
hdbscan_model = HDBSCAN(min_cluster_size=15, min_samples=1)
vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    max_df=0.8,
    min_df=2
)

representation_model = KeyBERTInspired()

topic_model = BERTopic(
    # embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    representation_model=representation_model,
)

In [63]:
topics, probs = topic_model.fit_transform(docs)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [64]:
topics = topic_model.reduce_outliers(docs, topics)
topic_model.reduce_topics(docs, nr_topics=20)
topics = topic_model.topics_
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,7937,-1_vibecoding ai_ai vibecoding_vibecoding vibe...,"[vibecoding ai, ai vibecoding, vibecoding vibe...",[apparently claude code has gotten worse and p...
1,0,9395,0_vibecoding app_vibecoding just_vibecoding vi...,"[vibecoding app, vibecoding just, vibecoding v...",[why are we only talking about vibecoding and ...
2,1,1960,1_vibecoding games_vibecoding ai_ai vibecoding...,"[vibecoding games, vibecoding ai, ai vibecodin...",[building has made us rethink how games are ma...
3,2,177,2_troll_meme coin_meme_memecoins,"[troll, meme coin, meme, memecoins, just meme,...",[good day for troll keep in mind this meme has...
4,3,156,3_vibecoding saas_saas vibecoding_vibecode saa...,"[vibecoding saas, saas vibecoding, vibecode sa...",[if you are using a saas that does not an ai y...
5,4,118,4_vibecoding apple_aipowered vibecoding_platfo...,"[vibecoding apple, aipowered vibecoding, platf...",[apple anthropic team up to build aipowered vi...
6,5,111,5_bot vibecoding_telegram bot_vibecode trading...,"[bot vibecoding, telegram bot, vibecode tradin...",[live vibecoding build your first telegram bot...
7,6,94,6_codexero cluster_codexero_codex_ideas onchain,"[codexero cluster, codexero, codex, ideas onch...",[codexero is a vibecoding ai dapp builder for ...
8,7,93,7_war_propaganda_military_nazi,"[war, propaganda, military, nazi, peace, gener...",[russian colonel confirming what everyone alre...
9,8,83,8_physicians_doctors_medical_doctor,"[physicians, doctors, medical, doctor, clinica...",[doctors building private practices is actuall...


In [65]:
#get topic from 1 to 10
for i in range(11):
  print(topic_model.get_topic(i))

[('vibecoding app', np.float32(0.79303)), ('vibecoding just', np.float32(0.74788946)), ('vibecoding vibecoding', np.float32(0.74254173)), ('vibecoded', np.float32(0.684194)), ('vibe', np.float32(0.5862783)), ('apps', np.float32(0.40671143)), ('coding', np.float32(0.36012986)), ('app', np.float32(0.35297972)), ('claude code', np.float32(0.33913934)), ('code', np.float32(0.32614124))]
[('vibecoding games', np.float32(0.73898584)), ('vibecoding ai', np.float32(0.7139829)), ('ai vibecoding', np.float32(0.7073007)), ('vibecoding game', np.float32(0.70287526)), ('ai studio', np.float32(0.5689125)), ('ai tools', np.float32(0.4991032)), ('ai coding', np.float32(0.49219054)), ('use ai', np.float32(0.4919741)), ('ai agents', np.float32(0.46291143)), ('using ai', np.float32(0.44484133))]
[('troll', np.float32(0.5489614)), ('meme coin', np.float32(0.48397094)), ('meme', np.float32(0.45518616)), ('memecoins', np.float32(0.44218346)), ('just meme', np.float32(0.42837158)), ('memecoin', np.float32(0.

In [66]:
topic_model.get_document_info(docs)

,Document,Topic,Name,Representation,Representative_Docs,Top_n_words,Probability,Representative_document
0,having thorough logging makes vibecoding x mor...,0,0_vibecoding app_vibecoding just_vibecoding vi...,"[vibecoding app, vibecoding just, vibecoding v...",[why are we only talking about vibecoding and ...,vibecoding app - vibecoding just - vibecoding ...,0.652435,False
1,i was antivibecoding but after using claude co...,-1,-1_vibecoding ai_ai vibecoding_vibecoding vibe...,"[vibecoding ai, ai vibecoding, vibecoding vibe...",[apparently claude code has gotten worse and p...,vibecoding ai - ai vibecoding - vibecoding vib...,0.000000,False
2,vibecoding is so last year for my followers fo...,0,0_vibecoding app_vibecoding just_vibecoding vi...,"[vibecoding app, vibecoding just, vibecoding v...",[why are we only talking about vibecoding and ...,vibecoding app - vibecoding just - vibecoding ...,1.000000,False
3,vibecoding is handsdown the best way to moneti...,-1,-1_vibecoding ai_ai vibecoding_vibecoding vibe...,"[vibecoding ai, ai vibecoding, vibecoding vibe...",[apparently claude code has gotten worse and p...,vibecoding ai - ai vibecoding - vibecoding vib...,0.000000,False
4,the number one tip i can give revibecoding vib...,0,0_vibecoding app_vibecoding just_vibecoding vi...,"[vibecoding app, vibecoding just, vibecoding v...",[why are we only talking about vibecoding and ...,vibecoding app - vibecoding just - vibecoding ...,0.694700,False
...,...,...,...,...,...,...,...,...
20506,yes this get the first version from them and d...,0,0_vibecoding app_vibecoding just_vibecoding vi...,"[vibecoding app, vibecoding just, vibecoding v...",[why are we only talking about vibecoding and ...,vibecoding app - vibecoding just - vibecoding ...,0.771948,False
20507,thats how i vibecode btw this is true prompt e...,0,0_vibecoding app_vibecoding just_vibecoding vi...,"[vibecoding app, vibecoding just, vibecoding v...",[why are we only talking about vibecoding and ...,vibecoding app - vibecoding just - vibecoding ...,0.209844,False
20508,just dropped a brandnew basketball shooter bui...,1,1_vibecoding games_vibecoding ai_ai vibecoding...,"[vibecoding games, vibecoding ai, ai vibecodin...",[building has made us rethink how games are ma...,vibecoding games - vibecoding ai - ai vibecodi...,1.000000,False
20509,the programmers have discovered intuition and ...,0,0_vibecoding app_vibecoding just_vibecoding vi...,"[vibecoding app, vibecoding just, vibecoding v...",[why are we only talking about vibecoding and ...,vibecoding app - vibecoding just - vibecoding ...,0.562874,False


In [67]:
topic_words = topic_model.get_topics()
topic_words = [[word for word, _ in topic_model.get_topic(topic_id)] for topic_id in topic_model.get_topics().keys() if topic_id != -1]

# Prepare docs for coherence model
tokenized_docs = [doc.split() for doc in docs]
dictionary = Dictionary(tokenized_docs)
corpus = [dictionary.doc2bow(text) for text in tokenized_docs]

# Get topic words
topic_words_raw = [[word for word, _ in topic_model.get_topic(topic_id)] for topic_id in topic_model.get_topics().keys() if topic_id != -1]

# Process topic_words_raw to get individual words that exist in our dictionary
processed_topic_words = []
for topic_list in topic_words_raw:
    current_topic_words = []
    for word_or_ngram in topic_list:
        # Split n-grams into individual words
        individual_words = word_or_ngram.split()
        for word in individual_words:
            # Only add words that are in our dictionary
            if word in dictionary.token2id:
                current_topic_words.append(word)
    # Only append non-empty topic lists
    if current_topic_words:
        processed_topic_words.append(current_topic_words)

# Calculate C_V coherence
coherence_model_cv = CoherenceModel(topics=processed_topic_words, texts=tokenized_docs, dictionary=dictionary, coherence='c_v')
coherence_cv = coherence_model_cv.get_coherence()
print(f"Coherence (C_V): {coherence_cv}")


# Calculate NPMI coherence
coherence_model_npmi = CoherenceModel(topics=processed_topic_words, texts=tokenized_docs, dictionary=dictionary, coherence='c_npmi')
coherence_npmi = coherence_model_npmi.get_coherence()
print(f"Coherence (NPMI): {coherence_npmi}")

Coherence (C_V): 0.5102728769205183
Coherence (NPMI): 0.11698687768227266
